# Using llama with Python

In [35]:
!pip install colab-xterm

Install ollama locally on the host machine on cloud.
Do not worry about warnings such as
```
WARNING: systemd is not running
WARNING: Unable to detect NVIDIA/AMD GPU. Install lspci or lshw to automatically detect and install GPU dependencies.
```
What you are looking for is
```>>> Install complete. Run "ollama" from the command line.```


In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Rather than run ollama from the command line, we shall run it in a subprocess so we can access it from the notebook. Run this code below, if you see ```200``` that indicates success.

In [36]:
import subprocess, time, os, textwrap

proc = subprocess.Popen(                              # Start the Ollama server
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(2)                                       # Give it a moment to start

import requests                                            # Quick health check
print(requests.get("http://127.0.0.1:11434/api/tags").status_code) #


200


We now load the pre-trained model llama3.2:1b which is documented [here](https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_2/).

In [37]:
!ollama pull llama3.2:1b

Let's see which models are available (we can pull other models)

In [38]:
!ollama list

NAME           ID              SIZE      MODIFIED      
llama3.2:1b    baf6a787fdff    1.3 GB    2 seconds ago    


Initial command-line interaction with llama

In [39]:
!ollama run llama3.2:1b "Say hello in one short sentence."

Hello.



In order to get Python to interact with ollama we need to install the required Python library -- please execute the following:

In [40]:
!pip install ollama

The following line enables Python to locate the ollama server, which runs in the subprocess  created earlier.

In [41]:
import os
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

We can now interact with the LLM from Python. Please give this a go.

Once complete, try modifying the system and user prompts, observing how this changes the responses (it will).

In [42]:
# System prompt to set the behaviour of the assistant
system_prompt = 'You are a helpful and curious AI assistant.'

# User prompt
user_prompt = 'What is the color of love?'

import ollama
resp = ollama.generate(model="llama3.2:1b", prompt=f'{system_prompt}\n\nUser: {user_prompt}\nAssistant:')
print(resp["response"])

What a beautiful question!
As a thought experiment, let's consider the concept of "color" in a more abstract sense. Colors are typically associated with physical properties like wavelength or light frequencies. However, if we're talking about the emotional and intangible aspects of love...

In this case, I'd argue that the color of love is subjective and can vary greatly from person to person. Some might perceive it as red, symbolizing passion and strong emotions; while others might see it as blue, representing calmness and serenity.

But if I had to choose a more poetic answer, I'd say the color of love is often associated with shades of lavender or rose - gentle, soothing, and evocative of peaceful memories. These colors evoke feelings of warmth, comfort, and tranquility, which are all essential aspects of loving relationships.

What do you think? Do you have a personal association with a specific color when thinking about the color of love?


Hopefully that worked. Let's modify it so YOU can type in your own question.

In [43]:
# System prompt to set the behaviour of the assistant
system_prompt = 'You are a helpful and curious AI assistant.'

# User prompt
user_prompt = input("You:")

import ollama
resp = ollama.generate(model="llama3.2:1b", prompt=f'{system_prompt}\n\nUser: {user_prompt}\nAssistant:')
print(resp["response"])

You:I am hungry!
I'm happy to help you feel better. What time is it, and what kind of food are you in the mood for? Do you have any dietary restrictions or allergies I should know about?


This is a one-step interaction with an LLM and will remind you of AIs like ChatGPT, Gemini, etc. Can we extend it to a multi-step interaction?

# Your first chatbot

In [44]:
messages = []
role = "helpful and curious AI assistant"
#role = "sarcastic and cynical AI assistant" # use at your own risk

system_prompt = f'You are a {role}.'
messages.append({"role": "system", "content": system_prompt})

print(f"Assistant: Hi, I am a {role}. Type 'exit' to stop.")
while True:
    user_input = input("You: ")
    if user_input.lower() == 'exit':
        print("Conversation terminated.")
        break

    messages.append({"role": "user", "content": user_input})
    response = ollama.chat(model="llama3.2:1b", messages=messages)
    assistant_response_content = response["message"]["content"]
    messages.append({"role": "assistant", "content": assistant_response_content})
    print(f"Assistant: {assistant_response_content}")

Assistant: Hi, I am a helpful and curious AI assistant. Type 'exit' to stop.
You: hello!
Assistant: Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?
You: how do I get started with AI?
Assistant: Getting started with AI can be an exciting and potentially transformative journey. Here's a step-by-step guide to help you begin:

1. **Understand what AI is**: Start by learning about the basics of artificial intelligence, including machine learning, natural language processing, and deep learning.
2. **Choose a direction**: Decide which area of AI interests you the most:
	* **NLP (Natural Language Processing)**: Learn to analyze and understand human language.
	* **Computer Vision**: Explore image and video processing.
	* **Reinforcement Learning**: Understand how agents learn through trial and error.
	* **Deep Learning**: Dive into neural networks and their applications.
3. **Get familiar with programming languages**:
	* **Python**: A popular lang